In [1]:
import jax.numpy as jnp
import numpy as np
import polars as pl
import polars.selectors as cs


In [2]:
excel_path = "data/bronze/uta_gas_usage.xlsx"

In [3]:
engine = pl.lazyframe.GPUEngine(raise_on_fail=True)

In [4]:
schema = {
    "Date": pl.Date,
    "Nom": pl.Float64,
    "Delivery": pl.Float64,
    "Usage - 1": pl.Float64,
    "Usage - 2": pl.Float64,
    "Usage - 2_1": pl.Float64,
}

In [5]:
usage_data_sheets = pl.read_excel(excel_path, sheet_id=0, schema_overrides=schema, columns=schema.keys())

In [6]:
num_records = sum(df.shape[0] for df in usage_data_sheets.values())
print(f"Total number of records across all sheets: {num_records}")

Total number of records across all sheets: 3524


In [7]:
for sheet_name, df in usage_data_sheets.items():
    data = df.with_columns(pl.lit(sheet_name).alias("Sheet Name"))
    usage_data_sheets[sheet_name] = data

In [8]:
usage_data = pl.concat(usage_data_sheets.values())
usage_data.shape

(3524, 7)

In [9]:
usage_data_clean = usage_data.filter(~(pl.all_horizontal(cs.float().is_null()) | pl.col("Date").is_null()))
usage_data_clean.shape

(3410, 7)

In [10]:
usage_data_clean.drop("Sheet Name").write_parquet("data/silver/uta_gas_usage.parquet")